# ResNet

核心思想是构造嵌套的函数块，只有当复杂的函数类包含较小的函数类的时候，才能确保他们的性能，对于深度神经网络，将新添加的层训练成恒等映射$f(x)=x$,原来的模型和现在的模型一样有效。同时，新模型可能得到更优的解来拟合训练数据集

## 残差块
假设原始输入x，希望学到的理想映射为f(x)

串联一个层改变函数的类，残差块加入到快速通道得到$f(x) = x + g(x)$

这里 $x$ 是输入， $g(x)$ 是两层卷积层要学习的内容（残差）。

为什么要多此一举加个 $+x$？

假设我们已经训练好了一个 20 层的网络，效果很完美。现在我们要把它加到 52 层。
对于新加的 32 层，理想情况下，它们最好什么都不做（也就是实现一个“恒等映射” $f(x) = x$），把前 20 层的完美结果原封不动传过去就行了。

在普通网络里：新加的卷积层要通过复杂的参数配合，精准学出 $f(x) = x$，这在线性激活和权重矩阵里是极其困难的。

在残差网络里：因为自带了 $+x$，新加的卷积层 $g(x)$ 只需要把所有权重参数学成 0（让 $g(x) = 0$ ），输出就自然变成了 $f(x) = x + 0 = x$。在神经网络里，配合 L2 正则化（权重衰减），把参数推向 0 是最容易不过的事情了。

### 反向传播时防止梯度消失

反向传播时，梯度流的“保底快速通道”从反向传播和梯度的角度来看，残差网络直接消灭了“梯度消失”。当我们要把下游的损失梯度传回上游时，对 $f(x) = x + g(x)$ 关于 $x$ 求导：$$\frac{\partial f(x)}{\partial x} = 1 + \frac{\partial g(x)}{\partial x}$$常数1就是 ResNet 的精妙之处！在普通深层网络中，梯度是靠一重重的矩阵连乘传回去的，只要中间有几层稍微小了一点，乘到最底层时梯度就变成了 0（梯度消失）。而在 ResNet 里，因为有一个 1 + ...，哪怕卷积层 $g(x)$ 的梯度彻底流失、缩减成了 0，依然能依靠这个保底的 1，把梯度百分之百、无损地送回最底层的网络。这就让训练 100 层甚至 1000 层的网络变成了可能。

### 代码实现

In [4]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l


class Residual(nn.Module):  #@save
    def __init__(self, input_channels, num_channels,
                 use_1x1conv=False, strides=1):  # 是否使用1*1卷积层，input_channels,num_channels表示输入通道数和输出通道数，strides表示步长
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, num_channels,
                               kernel_size=3, padding=1, stride=strides)  # 第一个卷积层保证数据的高宽不变
        self.conv2 = nn.Conv2d(num_channels, num_channels,
                               kernel_size=3, padding=1)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels,
                                   kernel_size=1, stride=strides)  # 使用1*1卷积层，就是为了把input_channels变换为output通道数，只是做了通道的融合处理
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(num_channels)  # 第一个卷积层的BatchNorm层，为了保持数据的高宽不变，BatchNorm层设置的通道数就是特征维度
        self.bn2 = nn.BatchNorm2d(num_channels)  # 第二个卷积层的BatchNorm层，为了保持数据的高宽不变

    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return F.relu(Y)

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


这里我们看一下，输入和输出是否一致

In [5]:
blk = Residual(3,3)  # 默认 1*1 卷积层关闭
X = torch.rand(4, 3, 6, 6)
Y = blk(X)
Y.shape  # 通过残差层发现高宽以及通道数都没有发生变化

torch.Size([4, 3, 6, 6])

增加输出通道数的同时，减半输出的高和宽

In [6]:
blk = Residual(3, 6, use_1x1conv=True, strides=2)
blk(X).shape  # X = torch.rand(4, 3, 6, 6) 这里面通道数从3变成了6，高宽从6变成了3

torch.Size([4, 6, 3, 3])

## ResNet模型

In [ ]:
b1 = nn.Sequential(nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),
                   nn.BatchNorm2d(64), nn.ReLU(),
                   nn.MaxPool2d(kernel_size=3, stride=2, padding=1)) # 第一个stage，这里面的设置做了两次减半

def resnet_block(input_channels, num_channels, num_residuals,
                 first_block=False):  # 定义了一个stage 特别判定第一个block，构造num_residuals个残差层
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            blk.append(Residual(input_channels, num_channels,
                                use_1x1conv=True, strides=2))  # 第一个stage，需要有1*1的卷积块
        else:
            blk.append(Residual(num_channels, num_channels))
    return blk

b2 = nn.Sequential(*resnet_block(64, 64, 2, first_block=True))
b3 = nn.Sequential(*resnet_block(64, 128, 2))
b4 = nn.Sequential(*resnet_block(128, 256, 2))
b5 = nn.Sequential(*resnet_block(256, 512, 2))

net = nn.Sequential(b1, b2, b3, b4, b5,
                    nn.AdaptiveAvgPool2d((1,1)),  #是一个自适应平均池化层，它的特点是：无论输入特征图的尺寸是多少，都能输出指定尺寸的特征图。
# 这里的每一个通道的特征的高宽都被球类全局平均池化
                    nn.Flatten(), nn.Linear(512, 10))

*看一下整个网络对输入数的变化*

In [ ]:
X = torch.rand(size= (1, 1, 224, 224))
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__, X.shape)


Sequential torch.Size([1, 64, 56, 56])
Sequential torch.Size([1, 64, 56, 56])
Sequential torch.Size([1, 128, 28, 28])
Sequential torch.Size([1, 256, 14, 14])
Sequential torch.Size([1, 512, 7, 7])
AdaptiveAvgPool2d torch.Size([1, 512, 1, 1])
Flatten torch.Size([1, 512])
Linear torch.Size([1, 10])


: 

## ResNet是怎么做到训练1000层网络的

在反向传播的时候，梯度从相乘变成相加，所以不会出现梯度消失